In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# Loan Approval Prediction - Exploratory Data Analysis\n", "Professional EDA: shape, types, missing values, target distribution, categorical vs target, distributions, correlation, and IQR outlier analysis."]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "from pathlib import Path\n",
    "sys.path.insert(0, str(Path.cwd().parent))\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "from src.utils import load_dataset\n",
    "from src.data_preprocessing import clean_dataframe, analyze_outliers, iqr_bounds\n",
    "sns.set(style='whitegrid')\n",
    "df = load_dataset()\n",
    "print(df.shape)\n",
    "df.head()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": ["df.info()\n", "df.dtypes"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": ["# Missing values\n", "df.isnull().sum()"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": ["# Target distribution\n", "sns.countplot(x='Loan_Status', data=df)\n", "plt.title('Approved (Y) vs Rejected (N)')\n", "plt.show()\n", "df['Loan_Status'].value_counts(normalize=True)"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Categorical features vs Loan_Status\n",
    "for col in ['Gender','Married','Education','Self_Employed','Credit_History','Property_Area']:\n",
    "    plt.figure(figsize=(5,3))\n",
    "    sns.countplot(x=col, hue='Loan_Status', data=df)\n",
    "    plt.title(f'{col} vs Loan_Status')\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Income and Loan Amount distributions\n",
    "for col in ['ApplicantIncome','LoanAmount']:\n",
    "    plt.figure(figsize=(5,3))\n",
    "    sns.histplot(df[col].dropna(), kde=True)\n",
    "    plt.title(f'{col} distribution')\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Correlation heatmap (numeric)\n",
    "cleaned = clean_dataframe(df)\n",
    "num = cleaned.select_dtypes(include=[np.number])\n",
    "plt.figure(figsize=(8,6))\n",
    "sns.heatmap(num.corr(), annot=True, cmap='coolwarm')\n",
    "plt.title('Correlation Heatmap')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## IQR Outlier Detection\n", "IQR = Q3 - Q1. Lower bound = Q1 - 1.5*IQR, Upper bound = Q3 + 1.5*IQR. Values outside bounds are outliers."]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "for col in ['ApplicantIncome','CoapplicantIncome','LoanAmount']:\n",
    "    q1,q3,iqr,low,up = iqr_bounds(pd.to_numeric(df[col], errors='coerce').dropna())\n",
    "    print(f'{col}: Q1={q1:.1f} Q3={q3:.1f} IQR={iqr:.1f} lower={low:.1f} upper={up:.1f}')\n",
    "analyze_outliers(clean_dataframe(df))"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["**Outlier decision:** Income and LoanAmount are right-skewed with genuine high earners. Outliers are **retained** (not deleted) because they represent real applicants; tree-based models handle them well, and StandardScaler reduces their dominance for linear/SVM models."]
  }
 ],
 "metadata": {
  "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
  "language_info": {"name": "python", "version": "3.10"}
 },
 "nbformat": 4,
 "nbformat_minor": 5
}